# Tensors, Autograd, And Probability

A language model is a differentiable function that assigns probabilities to discrete next-token events. Before we build the model, we need the tensor algebra and probability identities that make those probabilities trainable.


## Problem: What Must This Component Compute?

At this point in the model, the input is a batch of logits with shape `(B, T, V)`: batch size, sequence length, and vocabulary size. The component must convert those scores into a normalized categorical distribution and a scalar learning signal without leaking information about how the normalization was implemented.

We will first write the operation directly with tensors so every dimension is visible. Only after the numerical checks pass will we wrap the operation in a reusable module.

Why this matters later: when an architecture paper claims to improve attention, memory, quantization, or world modeling, it is usually changing one of these constraints: what information can move across positions, how much memory is required, what objective is optimized, or which representations are preserved.


## Notation, Shapes, And Factorization

Let a token sequence be `x_1, ..., x_T`. Autoregressive modeling writes the joint probability as

\[
p(x_1, \ldots, x_T) = \prod_{t=1}^{T} p(x_t \mid x_{<t}).
\]

Each conditional distribution is categorical over the vocabulary. A model therefore emits logits `z_t \in \mathbb{R}^V`, then converts them into probabilities with softmax.

Broadcasting matters because tensor code rarely loops over positions explicitly. If a tensor has shape `(B, T, V)` and we add a bias of shape `(V,)`, PyTorch implicitly copies that bias across the batch and time dimensions.


In [ ]:
import torch

features = torch.tensor([[1.0, 0.0, -1.0], [0.5, 0.5, 0.5]])
bias = torch.tensor([0.25, -0.25, 0.5])
shifted = features + bias

probs = torch.tensor([0.2, 0.3, 0.5])
values = torch.tensor([1.0, 2.0, 4.0])
expected_value = (probs * values).sum()

shifted, expected_value


In [ ]:
from llm_from_scratch.functional import stable_softmax

logits = torch.tensor([[1000.0, 1001.0, 1002.0], [-2.0, 0.0, 2.0]])
probs = stable_softmax(logits)
shifted_probs = stable_softmax(logits - logits.max(dim=-1, keepdim=True).values)

assert torch.allclose(probs.sum(dim=-1), torch.ones(2))
assert torch.allclose(probs, shifted_probs)

probs


## Softmax, Numerical Stability, And Cross-Entropy

Softmax maps logits `z_i` to probabilities

\[
p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}.
\]

Subtracting the same constant `c` from every logit leaves the probabilities unchanged because the factor `e^{-c}` cancels from numerator and denominator. Choosing `c = \max_i z_i` keeps the exponentials in a safe numerical range.

The next-token loss for target class `y` is the negative log-likelihood

\[
L = -\log p_y = -z_y + \log \sum_j e^{z_j}.
\]


In [ ]:
import torch
from llm_from_scratch.functional import cross_entropy_from_logits, stable_softmax

logits = torch.tensor([[1.0, 2.0, 3.0]], requires_grad=True)
target = torch.tensor([2])
loss = cross_entropy_from_logits(logits, target)
loss.backward()
stable_softmax(logits.detach()), loss.item(), logits.grad


In [ ]:
import torch.nn.functional as F

probs = stable_softmax(logits.detach())
expected_grad = probs.clone()
expected_grad[0, target.item()] -= 1.0

assert torch.allclose(loss.detach(), F.cross_entropy(logits.detach(), target))
assert torch.allclose(logits.grad, expected_grad, atol=1e-6)

{"gradient": logits.grad, "expected": expected_grad}


## Abstraction Bridge

In a full model, these logits usually come from a linear projection applied to hidden states. PyTorch autograd then applies the chain rule through every upstream tensor operation automatically. The useful mental model is still the explicit one above: the loss only asks the target logit to move up relative to the others.


In [ ]:
sequence_logits = torch.tensor(
    [[[2.0, 0.0, -1.0], [0.5, 1.5, -0.5]]],
    requires_grad=True,
)
sequence_targets = torch.tensor([[0, 1]])

custom_loss = cross_entropy_from_logits(sequence_logits, sequence_targets)
reference_loss = F.cross_entropy(
    sequence_logits.view(-1, sequence_logits.size(-1)),
    sequence_targets.view(-1),
)

assert torch.allclose(custom_loss, reference_loss)
custom_loss.item(), reference_loss.item()


## Exercise Checkpoint 1

1. If logits have shape `(B, T, V)`, what shape does `logits.reshape(-1, V)` produce and why is that convenient for cross-entropy?
2. Explain in one sentence why `softmax(z)` and `softmax(z - max(z))` are identical distributions.


## Exercise Checkpoint 2

1. Starting from `L = -\log softmax(z)_y`, derive the sign of the gradient on the target logit `z_y`.
2. If one token in a sequence receives probability `0.01`, what happens to the loss contribution for that token compared with probability `0.5`?
